Stage 5
----


# PyPSA-BC : Model Builder

In [1]:
# Reload modules to pick up code changes
import sys
import importlib

# Remove cached modules
modules_to_reload = [
    'workflow',
    'workflow.scripts',
    'workflow.scripts.build_model',
    'pypsa_bc',
    'pypsa_bc.utils',
    'pypsa_bc.attributes_parser'
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]

print("✓ Modules reloaded")

✓ Modules reloaded


## Stage 1: Configuration

In [2]:
from pypsa_bc.attributes_parser import AttributesParser

cfg = AttributesParser()
print("\n" + "=" * 80)
print("CONFIGURATION LOADED")
print("=" * 80)
print(f"✓ Scenario: {cfg.get_scenario}")
print(f"✓ Snapshot: {cfg.snapshot}")
print(f"✓ Data available: check build output below")
print("=" * 80)


CONFIGURATION LOADED
✓ Scenario: Test
✓ Snapshot: ('2021-01-01 00:00:00', '2021-12-31 23:00:00')
✓ Data available: check build output below


## Stage 2: Load Disaggregation

In [3]:
import subprocess
import os
from pathlib import Path

print("\n" + "=" * 80)
print("FETCH BC HYDRO DATA")
print("=" * 80 + "\n")

# Check if data already exists
bch_path = Path(cfg.data_cfg["data"]["load"]["bch"] + "2021.xls")

if bch_path.exists():
    size_mb = bch_path.stat().st_size / (1024 * 1024)
    print(f"✓ Data already exists: {bch_path}")
    print(f"  Size: {size_mb:.2f} MB\n")
else:
    print(f"⬇️  Fetching BC Hydro load data...")
    print(f"  Target: {bch_path}\n")
    
    result = subprocess.run(
        ["python", "-m", "workflow.scripts.fetch_inputs", "--only", "bch_load_2021"],
        cwd=os.getcwd(),
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.returncode != 0:
        print(f"⚠️  Error: {result.stderr}")
    else:
        print(f"✓ Data fetch completed\n")


FETCH BC HYDRO DATA

✓ Data already exists: data/downloaded_data/load/bc_hydro_load/BalancingAuthorityLoad2021.xls
  Size: 0.58 MB



In [4]:
import subprocess
import sys
import pandas as pd
from pathlib import Path
from workflow.scripts import disaggregate_load

print("\n" + "=" * 80)
print("LOAD DISAGGREGATION")
print("=" * 80 + "\n")

try:
    # Ensure xlrd is installed for .xls file reading
    try:
        import xlrd
    except ImportError:
        print("Installing xlrd for Excel support...")
        subprocess.run([sys.executable, "-m", "pip", "install", "xlrd", "--quiet"])
        import xlrd
    
    # Load BC Hydro balancing authority load data from config
    bch_load_path = Path(cfg.data_cfg["data"]["load"]["bch"] + "2021.xls")
    
    if not bch_load_path.exists():
        raise FileNotFoundError(
            f"BC Hydro load data not found at {bch_load_path}\n"
            f"Fetch it: python -m workflow.scripts.fetch_inputs --only bch_load_2021"
        )
    
    # Read raw load data (specify engine for .xls format)
    load_bch_raw: pd.DataFrame = pd.read_excel(bch_load_path, engine='xlrd')
    
    # Fix hourly load and get provincial total
    provincial_total_load_MWh: float = disaggregate_load.fix_hourly_load(load_bch_raw, 2021)
    provincial_total_load_MWh: float = provincial_total_load_MWh.LOAD.sum()  # MWh, provincial total
    
    print(f"✓ Provincial total load: {provincial_total_load_MWh:,.0f} MWh")
    print("✓ Running disaggregation...")
    
    # Disaggregate load to regions
    disaggregate_load.main(provincial_total_load_MWh)
    
    print("✓ Load disaggregation completed\n")
    
except FileNotFoundError as e:
    print(f"❌ Missing: {e}")
    
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

2026-07-26 10:54:19,890 - INFO - Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-07-26 10:54:19,890 - INFO - NumExpr defaulting to 16 threads.



LOAD DISAGGREGATION

Installing xlrd for Excel support...
❌ Error: ModuleNotFoundError: No module named 'xlrd'


/localhome/mei3/eliasinul/work/PyPSA_BC/.venv/bin/python: No module named pip
Traceback (most recent call last):
  File "/tmp/ipykernel_2806365/681476745.py", line 14, in <module>
    import xlrd
ModuleNotFoundError: No module named 'xlrd'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_2806365/681476745.py", line 18, in <module>
    import xlrd
ModuleNotFoundError: No module named 'xlrd'


## Stage 3: Build PyPSA Model

In [5]:
from workflow.scripts import build_model

print("\n" + "=" * 80)
print("BUILDING PyPSA MODEL")
print("=" * 80 + "\n")

try:
    # Build with parameters
    build_model.main(
        copperplate=False,                    # Regional mode
        capacity_choice="full_potential",
        tx_line_infinity=False,
        year=2021,
        include_vre_investments=False,

    )
    print("\n" + "=" * 80)
    print("✅ BUILD COMPLETED")
    print("=" * 80)
    
except FileNotFoundError as e:
    print(f"\n❌ Missing: {e}")
    print("Fetch data: python -m workflow.scripts.fetch_inputs")
    
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

2026-07-26 10:54:21,308 - WARNING - Importing network from PyPSA version v0.0.0 while current version is v1.2.4. Read the release notes at `https://go.pypsa.org/release-notes` to prepare your network for import.



BUILDING PyPSA MODEL

    ✖ Disclaimer: This model supports upto 28 Regional Districts (administrative regions) as nodes. The detailed loads (if provided) will be aggregated to these regional nodes.


2026-07-26 10:54:21,641 - WARNING - The following lines have buses which are not defined. Add them using n.add() or run n.sanitize() to add them automatically. Components with undefined buses:
Index(['32550'], dtype='object', name='name')
2026-07-26 10:54:21,644 - WARNING - The following lines have buses which are not defined. Add them using n.add() or run n.sanitize() to add them automatically. Components with undefined buses:
Index(['31834', '31843', '31862', '32059', '32225', '32226', '32242', '32552',
       '32764'],
      dtype='object', name='name')
2026-07-26 10:54:21,741 - INFO - Imported network 'Unnamed Network' has buses, lines, line_types, transformers, transformer_types
2026-07-26 10:54:23,401 - WARNING - The following links have buses which are not defined. Add them using n.add() or run n.sanitize() to add them automatically. Components with undefined buses:
Index(['BC_ALU_GSS Discharge Link'], dtype='object', name='name')
2026-07-26 10:54:23,418 - WARNING - The followin

    ✖ Creating BACKSTOP generators to the network...


2026-07-26 10:54:30,760 - WARNING - The attribute 'v_nom' is a standard attribute for other components but not for lines. This could cause confusion and it should be renamed. See also: https://go.pypsa.org/warning-attr-misleading.
2026-07-26 10:54:30,769 - WARNING - The attribute 'v_nom' is a standard attribute for other components but not for lines. This could cause confusion and it should be renamed. See also: https://go.pypsa.org/warning-attr-misleading.
2026-07-26 10:54:30,779 - WARNING - The attribute 'v_nom' is a standard attribute for other components but not for lines. This could cause confusion and it should be renamed. See also: https://go.pypsa.org/warning-attr-misleading.
2026-07-26 10:54:30,789 - WARNING - The attribute 'v_nom' is a standard attribute for other components but not for lines. This could cause confusion and it should be renamed. See also: https://go.pypsa.org/warning-attr-misleading.
2026-07-26 10:54:30,798 - WARNING - The attribute 'v_nom' is a standard attr

...

Solved in 325629 iterations and 74.26 seconds (109.24 work units)


2026-07-26 10:55:55,667 - INFO - Solved in 325629 iterations and 74.26 seconds (109.24 work units)


Optimal objective  3.286641812e+07


2026-07-26 10:55:55,668 - INFO - Optimal objective  3.286641812e+07
2026-07-26 10:56:11,165 - INFO -  Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 3118560 primals, 10030200 duals
Objective: 3.29e+07
Solver: gurobi
Runtime: 74.30s
Dual bound: 3.29e+07
Solver model: available
Solver message: 2

2026-07-26 10:56:11,268 - INFO - The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-p_set, Line-fix-s-lower, Line-fix-s-upper, Link-fix-p-lower, Link-fix-p-upper, Link-p-ramp_limit_up, Link-p-ramp_limit_down, Store-fix-e-lower, Store-fix-e-upper, Kirchhoff-Voltage-Law, Store-energy_balance were not assigned to the network.
/localhome/mei3/eliasinul/work/PyPSA_BC/.venv/lib/python3.12/site-packages/pypsa/network/io.py:1310: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=


✅ BUILD COMPLETED
